# Nouvelle partie pour la resolution de probleme coté new equation manager

In [ ]:
# Debug q_test instability with new EquationManager
import copy as cp
import jax.numpy as jnp

state_t = cp.deepcopy(sol)
params_t = {}

dt0 = hydrosim_test.timestep(state_t)
print("hydrosim_test dt0:", float(dt0))

prim0 = eq_test.get_primitives_from_conservatives(state_t)
print("eq_test primitive finite:", bool(jnp.isfinite(prim0).all()))
print("E_x(min,max):", float(prim0[0].min()), float(prim0[0].max()))
print("p(min,max):", float(prim0[4].min()), float(prim0[4].max()))
print("u(max abs):", float(jnp.abs(prim0[1]).max()))

try:
    (state1, params1), dt1 = hydrosim_test.hydrostep_adapt(0, (state_t, params_t), 0.0)
    print("step0 ok, dt1:", float(dt1), "finite state:", bool(jnp.isfinite(state1).all()))
except FloatingPointError as e:
    print("step0 failed with FloatingPointError:", e)

hydrosim_test dt0: 0.03513641655445099
eq_test primitive finite: True
E_x(min,max): 1.0 1.0
p(min,max): 0.09000000357627869 9.0
u(max abs): 0.0
step0 ok, dt1: 0.03513641655445099 finite state: True


In [ ]:
# Find first failing step for hydrosim_test and show RT speed mismatch indicators
import copy as cp
import jax.numpy as jnp

state_t = cp.deepcopy(sol)
params_t = {}
t_t = 0.0

for i in range(400):
    dt_i = float(hydrosim_test.timestep(state_t))
    prim_i = eq_test.get_primitives_from_conservatives(state_t)
    u_max = float(jnp.max(jnp.abs(prim_i[1])))
    # In LaxFriedrichs_Radiative_transfer, alpha ~= |u| + light_speed
    alpha_rt = u_max + float(eq_test.ligth_speed)
    cfl_eff = dt_i * alpha_rt

    if i % 20 == 0:
        print(f"i={i:03d} t={t_t:.4e} dt={dt_i:.4e} alpha_rt~{alpha_rt:.4e} dt*alpha_rt~{cfl_eff:.4e}")

    try:
        (state_t, params_t), dt_used = hydrosim_test.hydrostep_adapt(i, (state_t, params_t), t_t)
        t_t += float(dt_used)
    except FloatingPointError as e:
        print(f"FAIL at step {i}, t={t_t:.6e}, dt={dt_i:.6e}, dt*alpha_rt~{cfl_eff:.6e}")
        print("error:", e)
        break

print("done")

i=000 t=0.0000e+00 dt=3.5136e-02 alpha_rt~3.0000e+08 dt*alpha_rt~1.0541e+07
FAIL at step 4, t=3.513643e-02, dt=9.885230e-30, dt*alpha_rt~1.385115e-01
error: invalid value (nan) encountered in sub
done


In [ ]:
# Inspect hydrosim_test state evolution right before failure
import copy as cp
import jax.numpy as jnp

state_t = cp.deepcopy(sol)
params_t = {}
t_t = 0.0

for i in range(6):
    prim_i = eq_test.get_primitives_from_conservatives(state_t)
    finite_state = bool(jnp.isfinite(state_t).all())
    finite_prim = bool(jnp.isfinite(prim_i).all())
    E_min = float(prim_i[0].min())
    E_max = float(prim_i[0].max())
    p_min = float(prim_i[4].min())
    p_max = float(prim_i[4].max())
    u_max = float(jnp.max(jnp.abs(prim_i[1])))
    dt_i = float(hydrosim_test.timestep(state_t))

    print(f"i={i} finite_state={finite_state} finite_prim={finite_prim} t={t_t:.3e} dt={dt_i:.3e} E[min,max]=({E_min:.3e},{E_max:.3e}) p[min,max]=({p_min:.3e},{p_max:.3e}) u_max={u_max:.3e}")

    try:
        (state_t, params_t), dt_used = hydrosim_test.hydrostep_adapt(i, (state_t, params_t), t_t)
        t_t += float(dt_used)
    except FloatingPointError as e:
        print("FAILED at i=", i, "err:", e)
        break

i=0 finite_state=True finite_prim=True t=0.000e+00 dt=3.514e-02 E[min,max]=(1.000e+00,1.000e+00) p[min,max]=(9.000e-02,9.000e+00) u_max=0.000e+00
i=1 finite_state=True finite_prim=True t=3.514e-02 dt=3.569e-09 E[min,max]=(9.939e-01,1.006e+00) p[min,max]=(5.966e-21,7.683e+14) u_max=3.047e+06
i=2 finite_state=True finite_prim=True t=3.514e-02 dt=2.614e-09 E[min,max]=(9.384e-01,1.045e+00) p[min,max]=(5.630e-21,1.082e+15) u_max=1.230e+07
i=3 finite_state=True finite_prim=True t=3.514e-02 dt=2.933e-09 E[min,max]=(7.300e-01,1.209e+00) p[min,max]=(4.380e-21,6.339e+14) u_max=3.976e+07
i=4 finite_state=True finite_prim=True t=3.514e-02 dt=9.885e-30 E[min,max]=(1.000e-20,2.411e+00) p[min,max]=(6.000e-41,7.610e+14) u_max=1.401e+28
FAILED at i= 4 err: invalid value (nan) encountered in sub


## Carte mentale du pipeline DiffHydro (ordre d'appel Python)


Objectif: voir qui appelle quoi, dans quel ordre, et d'ou viennent les fonctions.



### Vue d'ensemble (lecture Python dans ce notebook)



1. Construction des objets (Cellule 5)

   - eq = dh.equationmanager.EquationManager()

   - eq_test = EquationManager_RT(...)

   - solver = dh.LaxFriedrichs_safe(...)

   - solver_test = dh.LaxFriedrichs_Radiative_transfer(...)

   - cf = dh.ConvectiveFlux(...)

   - cf_test = dh.ConvectiveFlux_Radiative_transfer(...)

   - hydrosim = dh.hydro(...)

   - hydrosim_test = dh.hydro(...)



2. Lancement simulation (Cellule 6)

   - hydrosim.evolve_till_time(...)

   - hydrosim_test.evolve_till_time(...)



3. Boucle interne de chaque pas (dans hydro_core)

   - timestep -> flux.timestep -> EquationManager.get_primitives_from_conservatives + EquationManager.get_signal_speed

   - hydrostep_adapt -> _hydrostep -> mol_solve_step -> integrator (RK2/SSPRK3/RK4)

   - integrator appelle rhs_unsplit

   - rhs_unsplit boucle sur x,y,z: boundary.impose -> ConvectiveFlux.flux

   - ConvectiveFlux.flux fait:

     - primitives = eq.get_primitives_from_conservatives

     - reconstruction L/R = recon.reconstruct_xi (PLM/MUSCL3)

     - conservatives L/R = eq.get_conservatives_from_primitives

     - solveur de Riemann = solver.solve_riemann_problem_xi

     - flux physiques = equation_manager.get_fluxes_xi

   - rhs = -div(F) puis update de l'etat par l'integrateur



### Qui fait quoi (resume court)



- EquationManager:

  - convertit primitives <-> conservatives

  - calcule flux physiques F(U)

  - calcule vitesses caracteristiques pour CFL



- Reconstruction (PLM/MUSCL3):

  - reconstruit les etats gauche/droite aux interfaces de maille



- Riemann solver (LaxFriedrichs...):

  - combine etat gauche/droite en flux numerique stable



- ConvectiveFlux:

  - assemble toute la chaine locale (conversion, reconstruction, Riemann, flux)



- hydro:

  - orchestre le temps (dt, boucle, integrateur, BC, divergence)


In [ ]:
# Arbre d'appel lisible + provenance des fonctions (a executer)

from pprint import pprint



call_tree = {

    "Notebook Cellule 5 (setup objets)": {

        "EquationManager (Euler)": "diffhydro/equationmanager.py::EquationManager",

        "EquationManager RT": "diffhydro/equationmanager_radiative_transf_no_chat.py::EquationManager",

        "Riemann solver (standard)": "diffhydro/solver/riemann_solver.py::LaxFriedrichs_safe",

        "Riemann solver (RT)": "diffhydro/solver/riemann_solver.py::LaxFriedrichs_Radiative_transfer",

        "ConvectiveFlux": "diffhydro/fluxes.py::ConvectiveFlux",

        "ConvectiveFlux RT": "diffhydro/fluxes.py::ConvectiveFlux_Radiative_transfer",

        "Hydro driver": "diffhydro/hydro_core.py::hydro",

    },

    "Notebook Cellule 6 (run)": {

        "hydrosim.evolve_till_time": {

            "timestep": {

                "flux.timestep": {

                    "get_primitives_from_conservatives": "EquationManager",

                    "get_signal_speed": "EquationManager",

                }

            },

            "hydrostep_adapt": {

                "_hydrostep": {

                    "mol_solve_step": {

                        "integrator (RK2/SSPRK3/RK4)": {

                            "rhs_unsplit": {

                                "for axis in x,y,z": {

                                    "boundary.impose": "diffhydro/boundary/boundary.py",

                                    "ConvectiveFlux.flux": {

                                        "get_primitives_from_conservatives": "EquationManager",

                                        "reconstruct_xi (PLM/MUSCL3)": "diffhydro/solver/recon.py",

                                        "get_conservatives_from_primitives": "EquationManager",

                                        "solve_riemann_problem_xi": "diffhydro/solver/riemann_solver.py",

                                        "get_fluxes_xi": "EquationManager",

                                    },

                                    "divergence des flux": "hydro_core.rhs_unsplit",

                                }

                            }

                        }

                    }

                }

            }

        }

    }

}



def print_tree(node, indent=0):

    pad = "  " * indent

    if isinstance(node, dict):

        for k, v in node.items():

            print(f"{pad}- {k}")

            print_tree(v, indent + 1)

    else:

        print(f"{pad}  -> {node}")



print("=== Carte mentale executable du pipeline DiffHydro ===")

print_tree(call_tree)


=== Carte mentale executable du pipeline DiffHydro ===
- Notebook Cellule 5 (setup objets)
  - EquationManager (Euler)
      -> diffhydro/equationmanager.py::EquationManager
  - EquationManager RT
      -> diffhydro/equationmanager_radiative_transf_no_chat.py::EquationManager
  - Riemann solver (standard)
      -> diffhydro/solver/riemann_solver.py::LaxFriedrichs_safe
  - Riemann solver (RT)
      -> diffhydro/solver/riemann_solver.py::LaxFriedrichs_Radiative_transfer
  - ConvectiveFlux
      -> diffhydro/fluxes.py::ConvectiveFlux
  - ConvectiveFlux RT
      -> diffhydro/fluxes.py::ConvectiveFlux_Radiative_transfer
  - Hydro driver
      -> diffhydro/hydro_core.py::hydro
- Notebook Cellule 6 (run)
  - hydrosim.evolve_till_time
    - timestep
      - flux.timestep
        - get_primitives_from_conservatives
            -> EquationManager
        - get_signal_speed
            -> EquationManager
    - hydrostep_adapt
      - _hydrostep
        - mol_solve_step
          - integrator (RK2

## Carte mentale detaillee: petites fonctions et interactions exactes


Cette section descend au niveau des petites fonctions utilisees pendant `evolve_till_time`.



### 1) Ordre d'appel global (un run)



1. Notebook construit les objets (`eq`, `solver`, `cf`, `hydrosim`).

2. `hydrosim.evolve_till_time(state, params, t_target)` demarre la boucle temporelle.

3. A chaque iteration:

   - `hydro.timestep` calcule `dt` via chaque flux/force.

   - `hydro._hydrostep` fait l'avancement d'un pas de temps.

   - `hydro.mol_solve_step` applique l'integrateur (`RK2`/`SSPRK3`/`RK4`).

   - l'integrateur appelle `hydro.rhs_unsplit` (RHS spatiale).

   - `rhs_unsplit` boucle sur les axes x,y,z:

     - `boundary.impose`

     - `ConvectiveFlux.flux`

     - calcul de divergence des flux.



### 2) Detail des petites fonctions (chaine ConvectiveFlux)



Pour chaque axe et chaque interface:



1. `EquationManager.get_primitives_from_conservatives(U)`

   - Convertit l'etat conservatif en primitives.

   - Sert a la reconstruction et au calcul des vitesses d'onde.



2. `Recon.reconstruct_xi(primitives, axis, j)` avec `j=0` (gauche) et `j=1` (droite)

   - Produit les etats d'interface L/R.

   - Utilise le limiteur choisi (`PLM`, `MUSCL3`, etc.).



3. `EquationManager.get_conservatives_from_primitives(...)`

   - Reconvertit les etats L/R en conservatives, necessaires au solveur de Riemann.



4. `RiemannSolver.solve_riemann_problem_xi(prim_L, prim_R, cons_L, cons_R, axis)`

   - Dispatch vers `_solve_riemann_problem_xi_single_phase`.

   - Calcule le flux numerique d'interface stable.



5. Dans le solveur (`LaxFriedrichs_safe` ou RT):

   - `equation_manager.get_fluxes_xi(...)` calcule les flux physiques gauche/droite.

   - calcule la vitesse numerique (`alpha`) via vitesse locale + celerite (son ou light_speed).

   - combine en flux Rusanov/Lax-Friedrichs.



6. `rhs_unsplit`: `rhs -= (F - roll(F))/dx`

   - Transforme les flux interfaces en derivee spatiale conservative.



7. Integrateur (`rk2_step` / `ssprk3_step` / `rk4_step`)

   - Fait les sous-etapes temporelles a partir de `rhs_unsplit`.



### 3) Interaction precise des managers (Euler vs RT)



- Euler (`equationmanager.py`):

  - variables actives: `[rho, vx, vy, vz, p]` en primitives.

  - `get_signal_speed` utilise la vitesse du son.



- RT no_chat (`equationmanager_radiative_transf_no_chat.py`):

  - variables actives: `[E_gamma, F_gamma_x, F_gamma_y, F_gamma_z]`.

  - `get_signal_speed` retourne une celerite basee sur `light_speed`.

  - le solveur RT doit recevoir des etats de taille coherente (4 actives ici).



### 4) Pourquoi chaque petite fonction existe



- Conversion primitive/conservative: separer physique thermodynamique et solveur numerique.

- Reconstruction: augmenter la precision spatiale aux interfaces.

- Riemann solver: imposer stabilite/causalite du flux numerique.

- Divergence des flux: appliquer la loi de conservation sur la maille.

- Integrateur en temps: controler precision/stabilite temporelle.


In [ ]:
# Documentation automatique fonction-par-fonction (roles + interactions)

import inspect

from types import MethodType



def where_is(func):

    try:

        f = inspect.getsourcefile(func)

        line = inspect.getsourcelines(func)[1]

        return f"{f}:{line}"

    except Exception:

        return "source non resolue"



def sig_of(func):

    try:

        return str(inspect.signature(func))

    except Exception:

        return "signature non resolue"



def safe_getattr(obj, name):

    try:

        return getattr(obj, name)

    except Exception:

        return None



def describe(owner_name, obj, name, role, called_by, calls, io_text):

    fn = safe_getattr(obj, name)

    print("=" * 110)

    print(f"{owner_name}.{name}")

    print(f"Role       : {role}")

    print(f"Called by  : {called_by}")

    print(f"Calls      : {calls}")

    print(f"I/O        : {io_text}")

    if fn is None:

        print("Signature  : <absente>")

        print("Source     : <absente>")

        return

    print(f"Signature  : {sig_of(fn)}")

    print(f"Source     : {where_is(fn)}")



print("\n########## A) HYDRO DRIVER ##########")

describe(

    "hydro", hydrosim, "evolve_till_time",

    role="Boucle temporelle jusqu'a t_target",

    called_by="Notebook (cellule run)",

    calls="hydrostep_adapt (dans une while_loop)",

    io_text="(fields0, params, t_target) -> (fields_f, params_f, t_f, dt_hist, n_steps)",

)

describe(

    "hydro", hydrosim, "hydrostep_adapt",

    role="Calcule dt puis avance un pas",

    called_by="evolve_till_time",

    calls="timestep, _hydrostep",

    io_text="(i, (fields, params), t) -> ((fields_new, params_new), dt)",

)

describe(

    "hydro", hydrosim, "timestep",

    role="CFL global via tous les flux/forces",

    called_by="hydrostep_adapt",

    calls="flux.timestep, force.timestep",

    io_text="fields -> dt",

)

describe(

    "hydro", hydrosim, "_hydrostep",

    role="Avancement numerique d'un pas (forcing + solveur spatial)",

    called_by="hydrostep_adapt",

    calls="forcing, mol_solve_step (ou sweep_stack)",

    io_text="(i, (fields, params), dt) -> (fields_new, params_new)",

)

describe(

    "hydro", hydrosim, "mol_solve_step",

    role="Integration temporelle via RHS unsplit",

    called_by="_hydrostep",

    calls="integrator(self.rhs_unsplit, ...)",

    io_text="(sol, dt, params) -> sol_new",

)

describe(

    "hydro", hydrosim, "rhs_unsplit",

    role="Assemble la divergence des flux sur x,y,z",

    called_by="integrator RK",

    calls="boundary.impose, flux(...), roll_with_halo",

    io_text="(sol, params) -> rhs",

)

describe(

    "hydro", hydrosim, "flux",

    role="Somme des contributions de tous les flux actifs",

    called_by="rhs_unsplit",

    calls="ConvectiveFlux.flux, ConductiveFlux.flux, ...",

    io_text="(sol, axis, params) -> F_total",

)



print("\n########## B) CONVECTIVE FLUX (STANDARD) ##########")

describe(

    "ConvectiveFlux", cf, "timestep",

    role="dt local CFL du terme convectif",

    called_by="hydro.timestep",

    calls="eq.get_primitives_from_conservatives, eq.get_signal_speed",

    io_text="sol -> dt_conv",

)

describe(

    "ConvectiveFlux", cf, "flux",

    role="Pipeline interface: prim -> reconstruction -> Riemann -> flux numerique",

    called_by="hydro.flux / hydro.rhs_unsplit",

    calls="eq conversions, recon.reconstruct_xi, solver.solve_riemann_problem_xi",

    io_text="(sol, ax, params, flux_prev) -> F_interface",

)

describe(

    "ConvectiveFlux", cf, "compute_positivity_preserving_interpolation",

    role="Corrige etats reconstruits non physiques",

    called_by="ConvectiveFlux.flux (si positivity=True)",

    calls="WENO1.reconstruct_xi, eq.get_conservatives_from_primitives",

    io_text="(primitives, primitives_xi_j, j, axis) -> (cons_xi_j, prim_xi_j, counter)",

)



print("\n########## C) EQUATION MANAGER (STANDARD) ##########")

describe(

    "EquationManager", eq, "get_primitives_from_conservatives",

    role="U -> W",

    called_by="flux.timestep, ConvectiveFlux.flux, diagnostics",

    calls="get_pressure/get_isothermal_pressure",

    io_text="conservatives -> primitives",

)

describe(

    "EquationManager", eq, "get_conservatives_from_primitives",

    role="W -> U",

    called_by="ConvectiveFlux.flux",

    calls="get_specific_energy",

    io_text="primitives -> conservatives",

)

describe(

    "EquationManager", eq, "get_fluxes_xi",

    role="Flux physique F(U) sur un axe",

    called_by="Riemann solver",

    calls="(calcul direct des composantes)",

    io_text="(primitives, conservatives, axis) -> flux_physique",

)

describe(

    "EquationManager", eq, "get_signal_speed",

    role="Celerite pour CFL",

    called_by="ConvectiveFlux.timestep",

    calls="get_speed_of_sound",

    io_text="(primitives, axis) -> a",

)

describe(

    "EquationManager", eq, "get_speed_of_sound",

    role="Vitesse du son",

    called_by="get_signal_speed + solveurs",

    calls="(calcul direct)",

    io_text="(p, rho) -> c_s",

)



print("\n########## D) RIEMANN SOLVER ##########")

describe(

    "RiemannSolver", solver, "solve_riemann_problem_xi",

    role="Entree generique, dispatch selon equation_type",

    called_by="ConvectiveFlux.flux",

    calls="_solve_riemann_problem_xi_single_phase",

    io_text="(prim_L, prim_R, cons_L, cons_R, axis) -> (F, aux1, aux2)",

)

describe(

    "LaxFriedrichs_safe", solver, "_solve_riemann_problem_xi_single_phase",

    role="Flux numerique type Rusanov stable",

    called_by="solve_riemann_problem_xi",

    calls="eq.get_fluxes_xi, eq.get_speed_of_sound",

    io_text="(W_L, W_R, U_L, U_R, axis) -> F_num",

)



print("\n########## E) RECONSTRUCTION ##########")

recon_obj = cf.recon

recon_name = type(recon_obj).__name__

describe(

    recon_name, recon_obj, "reconstruct_xi",

    role="Reconstruit etats gauche/droite aux interfaces",

    called_by="ConvectiveFlux.flux",

    calls="limiter si necessaire",

    io_text="(buffer, axis, j) -> etat_interface",

)



print("\n########## F) BRANCHE RT (si presente) ##########")

if 'cf_test' in globals() and 'eq_test' in globals() and 'solver_test' in globals():

    describe(

        "ConvectiveFlux_Radiative_transfer", cf_test, "flux",

        role="Pipeline de flux pour etat RT",

        called_by="hydrosim_test.rhs_unsplit",

        calls="eq_test conversions, recon, solver_test",

        io_text="(sol, ax, params, flux_prev) -> F_interface_RT",

    )

    describe(

        "EquationManager_RT", eq_test, "get_primitives_from_conservatives",

        role="U_RT -> W_RT",

        called_by="flux RT + diagnostics",

        calls="(calcul direct)",

        io_text="conservatives_RT -> primitives_RT",

    )

    describe(

        "EquationManager_RT", eq_test, "get_conservatives_from_primitives",

        role="W_RT -> U_RT",

        called_by="ConvectiveFlux_Radiative_transfer.flux",

        calls="(calcul direct)",

        io_text="primitives_RT -> conservatives_RT",

    )

    describe(

        "EquationManager_RT", eq_test, "get_fluxes_xi",

        role="Flux physique RT",

        called_by="LaxFriedrichs_Radiative_transfer",

        calls="(calcul direct)",

        io_text="(primitives_RT, conservatives_RT, axis) -> flux_RT",

    )

    describe(

        "EquationManager_RT", eq_test, "get_signal_speed",

        role="Celerite caracteristique RT",

        called_by="timestep RT",

        calls="get_speed_of_sound (RT/light_speed)",

        io_text="(primitives_RT, axis) -> a_RT",

    )

    describe(

        "LaxFriedrichs_Radiative_transfer", solver_test, "_solve_riemann_problem_xi_single_phase",

        role="Flux numerique RT diffuse",

        called_by="solve_riemann_problem_xi",

        calls="eq_test.get_fluxes_xi",

        io_text="(W_L_RT, W_R_RT, U_L_RT, U_R_RT, axis) -> F_RT",

    )

else:

    print("Branche RT non initialisee dans le kernel courant.")



########## A) HYDRO DRIVER ##########
hydro.evolve_till_time
Role       : Boucle temporelle jusqu'a t_target
Called by  : Notebook (cellule run)
Calls      : hydrostep_adapt (dans une while_loop)
I/O        : (fields0, params, t_target) -> (fields_f, params_f, t_f, dt_hist, n_steps)
Signature  : (input_fields, params, t_target: float, max_steps: int | None = None)
Source     : /mnt/data2/travail/stage/DiffHydro_public/examples/athena/../../diffhydro/hydro_core.py:449
hydro.hydrostep_adapt
Role       : Calcule dt puis avance un pas
Called by  : evolve_till_time
Calls      : timestep, _hydrostep
I/O        : (i, (fields, params), t) -> ((fields_new, params_new), dt)
Signature  : (i, state, current_time)
Source     : /mnt/data2/travail/stage/DiffHydro_public/examples/athena/../../diffhydro/hydro_core.py:358
hydro.timestep
Role       : CFL global via tous les flux/forces
Called by  : hydrostep_adapt
Calls      : flux.timestep, force.timestep
I/O        : fields -> dt
Signature  : (fields)